# 02 · Silver transactions
Converts bronze transactions to real types, checks conversion quality, and adds stable IDs.

In [0]:
from pyspark.sql import functions as F

BRONZE = "workspace.aml_bronze"
SILVER = "workspace.aml_silver"

bronze = spark.table(f"{BRONZE}.transactions")
def normalize_bank(col):
    # Canonical bank ID: remove leading zeros, keep a single "0" if all zeros
    return F.regexp_replace(col, r"^0+(?=.)", "")

In [0]:
typed = bronze.select(
    F.expr("try_to_timestamp(timestamp, 'yyyy/MM/dd HH:mm')").alias("txn_timestamp"),
    normalize_bank(F.trim("from_bank")).alias("from_bank"),
    F.trim("from_account").alias("from_account"),
    normalize_bank(F.trim("to_bank")).alias("to_bank"),
    F.trim("to_account").alias("to_account"),
    F.expr("try_cast(amount_received AS DECIMAL(24,8))").alias("amount_received"),
    F.trim("receiving_currency").alias("receiving_currency"),
    F.expr("try_cast(amount_paid AS DECIMAL(24,8))").alias("amount_paid"),
    F.trim("payment_currency").alias("payment_currency"),
    F.trim("payment_format").alias("payment_format"),
    F.expr("try_cast(is_laundering AS INT)").alias("is_laundering"),
    "_source_file",
)

typed.printSchema()

In [0]:
checks = typed.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("txn_timestamp").isNull().cast("int")).alias("bad_timestamp"),
    F.sum(F.col("amount_received").isNull().cast("int")).alias("bad_amount_received"),
    F.sum(F.col("amount_paid").isNull().cast("int")).alias("bad_amount_paid"),
    F.sum(F.col("is_laundering").isNull().cast("int")).alias("bad_label"),
)

display(checks)

In [0]:
display(
    typed.groupBy("is_laundering")
    .count()
    .orderBy("is_laundering")
)

In [0]:
keyed = (
    typed
    .withColumn("from_account_id", F.concat_ws("_", "from_bank", "from_account"))
    .withColumn("to_account_id", F.concat_ws("_", "to_bank", "to_account"))
    .withColumn(
        "is_self_transfer",
        F.col("from_account_id") == F.col("to_account_id"),
    )
)

In [0]:
all_accounts = (
    keyed.select(
        F.col("from_bank").alias("bank"),
        F.col("from_account").alias("account"),
    )
    .union(
        keyed.select(
            F.col("to_bank").alias("bank"),
            F.col("to_account").alias("account"),
        )
    )
)

display(
    all_accounts.agg(
        F.countDistinct("account").alias("by_account_number_only"),
        F.countDistinct("bank", "account").alias("by_bank_and_account"),
    )
)

In [0]:
display(
    keyed.groupBy("is_self_transfer", "payment_format")
    .count()
    .orderBy("is_self_transfer", "payment_format")
)

In [0]:
business_cols = [
    "txn_timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_received", "receiving_currency", "amount_paid",
    "payment_currency", "payment_format", "is_laundering",
]

dupes = (
    keyed.groupBy(*business_cols)
    .count()
    .filter("count > 1")
)

display(
    dupes.agg(
        F.count("*").alias("duplicate_groups"),
        F.sum("count").alias("rows_in_duplicate_groups"),
    )
)

In [0]:
display(dupes.orderBy(F.desc("count")).limit(10))

In [0]:
fact_cols = [
    "txn_timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_received", "receiving_currency", "amount_paid",
    "payment_currency", "payment_format",
]

hashed = keyed.withColumn(
    "txn_hash",
    F.substring(
        F.sha2(F.concat_ws("|", *[F.col(c).cast("string") for c in fact_cols]), 256),
        1,
        16,
    ),
)

In [0]:
from pyspark.sql import Window

dup_window = Window.partitionBy("txn_hash").orderBy("txn_hash")

with_ids = (
    hashed
    .withColumn("txn_seq", F.row_number().over(dup_window))
    .withColumn("txn_id", F.concat_ws("-", "txn_hash", F.col("txn_seq").cast("string")))
)

In [0]:
silver = with_ids.select(
    "txn_id",
    "txn_hash",
    "txn_timestamp",
    "from_account_id",
    "from_bank",
    "from_account",
    "to_account_id",
    "to_bank",
    "to_account",
    "amount_paid",
    "payment_currency",
    "amount_received",
    "receiving_currency",
    "payment_format",
    "is_self_transfer",
    "is_laundering",
    "_source_file",
    F.current_timestamp().alias("_processed_at"),
)

In [0]:
bronze_rows = bronze.count()
silver_rows = silver.count()
unique_ids = silver.select("txn_id").distinct().count()

print(f"Bronze rows: {bronze_rows:,}")
print(f"Silver rows: {silver_rows:,}")
print(f"Unique IDs:  {unique_ids:,}")

assert silver_rows == bronze_rows, "Rows were lost or added between bronze and silver"
assert unique_ids == silver_rows, "Transaction IDs are not unique"

print("Quality gate passed")

In [0]:
(
    silver.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{SILVER}.transactions")
)